# ตอนที่ 9: Benchmarking — ตัวเลข accuracy ที่ไม่มี confidence interval คือข่าวลือ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kobkrit/thai-llm-tutorials/blob/main/notebooks/09_benchmarking.ipynb)

โน้ตบุ๊กประกอบบทความ **[LLM 9/10] Benchmarking: ตัวเลข accuracy ที่ไม่มี confidence interval คือข่าวลือ**

โดย **ดร.กอบกฤตย์ วิริยะยุทธกร** — ผู้สร้าง OpenThaiGPT, CEO บริษัท iApp Technology
บทความฉบับเต็ม: [kobkrit.com](https://kobkrit.com/blog/llm-09-benchmarking)

---

## โครงของโน้ตบุ๊กนี้ (เรียงตรงกับหัวข้อในบทความ)

1. ปัญหา (Problem statement)
2. เราจะทำอะไร (Solution)
3. สมการ (Equation)
4. เห็นภาพสมการ (Visualize)
5. เตรียมสภาพแวดล้อม (Environment) — และวัด baseline ก่อนแตะอะไรทั้งสิ้น
6. เตรียมข้อมูล (Data) — **มีด่านตรวจ license ที่ห้ามข้าม**
7. โค้ดหลัก (Main code) — เขียนระบบให้คะแนน 3 โหมดจากศูนย์
8. ผลลัพธ์ (Results) — ทุก checkpoint บนแกนเดียว พร้อม Wilson CI
9. เปรียบเทียบ (Comparison) — โมเดลตัวเดิม คะแนนแกว่งกี่จุดตามวิธีวัด
10. สรุป (Summary)

---

## หัวใจของโน้ตบุ๊กนี้

ตอนนี้ **ไม่เทรนอะไรเลยแม้แต่ step เดียว** — และนั่นคือจุดเด่น ไม่ใช่จุดอ่อน
เราจะเขียนระบบให้คะแนน 3 โหมด (log-likelihood, generative exact-match, LLM-as-judge)
ขึ้นจากศูนย์ แล้วพิสูจน์ด้วยตัวเลขจริงว่า **การตั้งค่าที่ไม่มีใครรายงานใน paper
เปลี่ยนคะแนนของโมเดลตัวเดิมได้มากกว่าช่องว่างบน leaderboard ที่คนเถียงกัน**

ทุกตัวเลขในโน้ตบุ๊กนี้ใช้เครื่องมือสถิติจาก `kobeval` ตัวเดียวกับที่ใช้มาทั้งซีรีส์:
`wilson_ci`, `mcnemar`, `pass_at_k`, `th_ratio` และ `evaluate`

---

## ก่อนเริ่ม

- โน้ตบุ๊กนี้ออกแบบมาให้รันได้บน **Google Colab แบบฟรี (Tesla T4)**
  เลือก `Runtime > Change runtime type > T4 GPU` ก่อนรันเซลล์แรก
- T4 เป็นสถาปัตยกรรม Turing (sm_75) ซึ่ง **ไม่รองรับ bfloat16 และไม่รองรับ FlashAttention-2**
  เราจึงกำหนด `torch_dtype=torch.float16` และ `attn_implementation="sdpa"` เอง
  อย่างชัดเจนทุกครั้ง (รายละเอียดอยู่ในคอมเมนต์ของ Cell 0 — อ่านให้ครบ)
  ตอนนี้เป็น inference ล้วน ไม่มี optimizer จึงไม่ต้องยุ่งกับ `fp16=True` เลย
- ทุกตอนในซีรีส์นี้ใช้ชุดวัดผลกลางตัวเดียวกันคือ `kobeval` และเบนช์มาร์ก **KobEval-TH**
  (120 ข้อ, 4 slice) เพื่อให้ตัวเลขของแต่ละตอนเทียบกันได้จริง
- **ห้ามเทรนบน KobEval-TH เด็ดขาด** ชุดนี้เป็นชุดวัดผลอย่างเดียว
  ตอนนี้เราเพิ่มข้อสอบภายนอกอีกสองชุด: `scb10x/thai_exam` และ `VISAI-AI/gsm8k-thai`
  ซึ่งทั้งคู่ต้องผ่าน **ด่านตรวจ license** ในหัวข้อ 6 ก่อนใช้

## เวลาที่ใช้โดยประมาณบน T4 ฟรี (รวมเวลาโหลดโมเดล)

| ขั้นตอน | เวลาโดยประมาณ |
|---|---|
| Cell 0 ติดตั้ง dependencies | ~2 นาที |
| Cell 1 โหลดโมเดล + วัด baseline KobEval-TH ครบ 4 slice | ~9 นาที |
| ด่านตรวจ license + โหลดข้อสอบ | ~1 นาที |
| scorer 3 โหมด + กวาด checkpoint (log-likelihood เร็วมาก) | ~4 นาที |
| ตารางแกว่งคะแนน + pass@k + contamination scan | ~4 นาที |
| **รวม** | **~20 นาที** |

ถ้าเวลาไม่พอ: ลด `N_EXAM` / `N_GSM` ในหัวข้อ 6 หรือลด `KOBEVAL_SLICES` ใน Cell 1
เหลือ `["TH-KNOW"]` — สัญญาการวัดผลต่อข้อเหมือนเดิมทุกประการ


In [ ]:
# =============================================================================
# CELL 0 -- SETUP
# ชุดนี้ต้อง "เหมือนกันทุกตัวอักษร" (byte-identical) ในโน้ตบุ๊กทั้ง 10 ตอน
# ตรวจสอบด้วย: python3 scripts/check_cell0.py
# =============================================================================
# ทำไมต้องเหมือนกันทุกตอน: ถ้า setup ต่างกันแม้นิดเดียว ตัวเลขที่วัดได้จาก
# แต่ละตอนจะเปรียบเทียบกันไม่ได้ ซึ่งทำให้ทั้งซีรีส์นี้ไร้ความหมาย
# -----------------------------------------------------------------------------

# 1) ติดตั้ง dependencies (pin เวอร์ชันไว้ทั้งหมด เพื่อให้ผลลัพธ์ทำซ้ำได้)
#    หมายเหตุ: เราจงใจ "ไม่" ติดตั้ง torch ใหม่ เพราะ Colab มี torch ที่ build
#    มาให้ตรงกับ CUDA driver อยู่แล้ว การ pip install torch ทับคือสาเหตุอันดับ
#    หนึ่งที่ทำให้ notebook พังบน Colab
!pip install -q \
    "transformers==4.51.3" \
    "accelerate==1.6.0" \
    "datasets==3.5.0" \
    "peft==0.15.1" \
    "trl==0.16.1" \
    "bitsandbytes==0.45.5" \
    "sentencepiece==0.2.0" \
    "matplotlib==3.10.1"

# 2) ติดตั้งฟอนต์ไทยให้ matplotlib (ไม่งั้นกราฟจะขึ้นเป็นสี่เหลี่ยม tofu)
!apt-get install -y fonts-thai-tlwg > /dev/null 2>&1

# 3) ดึง repo (ได้ทั้งแพ็กเกจ kobeval และชุดข้อมูล KobEval-TH)
import os
REPO_DIR = "/content/thai-llm-tutorials"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/kobkrit/thai-llm-tutorials.git /content/thai-llm-tutorials
!pip install -q -e /content/thai-llm-tutorials

# ลดการแตกกระจายของหน่วยความจำ (fragmentation) -- ต้องตั้งก่อน torch แตะ CUDA
# T4 มี VRAM จำกัด พอ tensor ก้อนใหญ่ถูกจอง/คืนสลับกัน จะเกิดช่องว่างที่ใช้ต่อไม่ได้
# จน OOM ทั้งที่ยังเหลือที่รวม ๆ อยู่ (ข้อความ error ของ PyTorch เองก็แนะนำค่านี้)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import random
import numpy as np
import torch

# 4) ยืนยันว่ามี GPU จริง -- ถ้าไม่มีให้หยุดตรงนี้เลย ดีกว่าไปพังตอน train
assert torch.cuda.is_available(), (
    "ไม่พบ GPU! ไปที่ Runtime > Change runtime type > Hardware accelerator > GPU "
    "แล้วรันเซลล์นี้ใหม่"
)

GPU_NAME = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
# bf16 "แบบมีฮาร์ดแวร์รองรับจริง" เริ่มที่ Ampere (sm_80) ขึ้นไปเท่านั้น
#
# หมายเหตุสำคัญ: torch.cuda.is_bf16_supported() ตอบ True บน T4 (sm_75) ด้วย
# ตั้งแต่ torch รุ่นใหม่ ๆ เพราะมันนับ "การจำลอง (emulation)" ว่ารองรับด้วย
# ซึ่งช้ากว่า fp16 มากและไม่ใช่สิ่งที่เราต้องการ เราจึงเช็ค compute capability
# ตรง ๆ แทน -- นี่คือกับดักจริงที่เจอตอนรันบน Colab
NATIVE_BF16 = CAPABILITY[0] >= 8
SUPPORTS_BF16 = NATIVE_BF16          # ใช้ชื่อเดิมต่อได้ แต่ความหมายคือ "native"

print(f"GPU            : {GPU_NAME}")
print(f"Compute cap.   : sm_{CAPABILITY[0]}{CAPABILITY[1]}")
print(f"VRAM           : {VRAM_GB:.1f} GB")
print(f"SUPPORTS_BF16  : {SUPPORTS_BF16}   (native; torch บอกว่า "
      f"{torch.cuda.is_bf16_supported()} เพราะนับ emulation ด้วย)")
print(f"torch          : {torch.__version__}")

# -----------------------------------------------------------------------------
# 5) จุดสำคัญที่สุดของเซลล์นี้ -- อ่านให้ครบ
#
# ถ้าคุณได้ Tesla T4 (ซึ่งเป็นค่าปกติของ Colab ฟรี) คุณจะเห็น
#     SUPPORTS_BF16 : False   (native)
# ทั้งที่ torch.cuda.is_bf16_supported() ตอบ True -- อย่าเชื่อค่านั้น
#
# T4 เป็นสถาปัตยกรรม Turing (sm_75) ซึ่ง "ไม่รองรับ" สองอย่างนี้:
#   - bfloat16  -> ต้องใช้ float16 แทน
#   - FlashAttention-2 -> ต้องใช้ sdpa แทน
#
# กับดักที่คนเจอบ่อยที่สุด: config ของ Qwen3-0.6B ระบุ torch_dtype: bfloat16
# ไว้ในไฟล์ ดังนั้นถ้าเขียน torch_dtype="auto" transformers จะเชื่อ config
# แล้วโหลดเป็น bf16 บนการ์ดที่ไม่รองรับ ผลคือได้ NaN, loss ไม่ลด หรือ
# โมเดลพ่นข้อความมั่ว โดยไม่มี error ฟ้องให้เห็นเลย
#
# เราจึงกำหนดค่าพวกนี้เอง "อย่างชัดเจน" ทุกครั้ง ไม่พึ่ง auto
# -----------------------------------------------------------------------------
DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
ATTN_IMPL = "sdpa"          # ห้ามใช้ flash_attention_2 บน T4
USE_FP16 = not SUPPORTS_BF16  # ส่งเข้า TrainingArguments: fp16=USE_FP16, bf16=SUPPORTS_BF16

print(f"\n--> DTYPE      : {DTYPE}")
print(f"--> ATTN_IMPL  : {ATTN_IMPL}")
print(f"--> fp16={USE_FP16}, bf16={SUPPORTS_BF16}  (ใช้คู่นี้กับ TrainingArguments)")

# 6) ตั้ง seed ทุกตัวที่เกี่ยวข้อง เพื่อให้ผลลัพธ์ทำซ้ำได้
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# 7) import kobeval -- ชุดวัดผลกลางที่ใช้เหมือนกันทั้ง 10 ตอน
#    pip install -e ลงทะเบียนแพ็กเกจไว้แล้ว แต่ sys.path ของเคอร์เนลที่รันอยู่
#    อาจยังไม่เห็น จึงใส่ path ของ repo เข้าไปตรง ๆ กันพลาด (idempotent)
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from kobeval import evaluate, compare, plot_before_after, th_ratio, wilson_ci
from kobeval import EVAL_CONTRACT, verify_checksums
from kobeval.plotting import use_thai_font

# 8) ตั้งฟอนต์ไทยให้ matplotlib "ครั้งเดียวตรงนี้" มีผลกับทุกกราฟในโน้ตบุ๊ก
#    ทำไมต้องมีขั้นตอนนี้: apt ติดตั้งฟอนต์ใน (2) หลังจาก matplotlib สร้าง
#    แคชรายชื่อฟอนต์ไปแล้ว มันจึงมองไม่เห็นฟอนต์ไทย และวาดออกมาเป็นกล่อง []
#    use_thai_font() จะสแกนดิสก์ใหม่แล้วลงทะเบียนฟอนต์ให้
_thai_font = use_thai_font()
print(f"Thai font      : {_thai_font or 'ไม่พบ! กราฟภาษาไทยจะเป็นกล่อง [] -- รัน apt-get ในข้อ (2) ใหม่'}")

print(f"\nkobeval contract: {EVAL_CONTRACT}")
print(f"KobEval-TH checksum ok: {verify_checksums()['ok']}")


---

## 1. ปัญหา (Problem statement)

*"โมเดล X ได้ 71.2% บน ThaiExam แซงโมเดล Y ที่ได้ 69.8%"* — คำถามเดียวที่ควรถามคือ
**วัดจากกี่ข้อ** ถ้าชุดทดสอบมี 100 ข้อ ช่วงความเชื่อมั่น 95% ของแต่ละตัวเลขกว้างราว ±8–10 จุด
แปลว่า 71.2 กับ 69.8 คือ**ตัวเลขเดียวกัน**ที่บังเอิญสุ่มออกมาไม่เท่ากัน

และปัญหาไม่ได้หยุดที่ขนาดชุดทดสอบ เพราะ "คะแนน benchmark" หนึ่งตัวเกิดจาก
การตัดสินใจหลายชั้นที่แทบไม่มีใครรายงาน: ให้คะแนนแบบ log-likelihood หรือ generative,
normalize ความยาวตัวเลือกหรือไม่ ($\gamma$), 0-shot หรือ 5-shot, เปิด thinking mode ไหม,
และข้อสอบรั่วอยู่ในข้อมูลเทรนหรือเปล่า (contamination)

---

## 2. เราจะทำอะไร (Solution)

1. **เขียนระบบให้คะแนน 3 โหมดจากศูนย์** — log-likelihood multiple-choice,
   generative exact-match (พร้อม normalization ภาษาไทย), และ LLM-as-judge พร้อม rubric
2. **เอา checkpoint จากตอนก่อน ๆ มาวัดบนแกนเดียวกัน** ด้วย `scb10x/thai_exam`,
   `VISAI-AI/gsm8k-thai` และ KobEval-TH
3. **ติด Wilson 95% CI ให้ทุกตัวเลข** และใช้ McNemar เทียบคู่ที่คะแนนใกล้กันที่สุด
4. **สแกน contamination** ระหว่างข้อสอบกับคลัง CPT ของตอนที่ 1 ด้วย character n-gram

> **แนวคิดหลัก:** คะแนน benchmark ไม่ใช่คุณสมบัติของโมเดล มันคือคุณสมบัติของ
> **(โมเดล × วิธีวัด × ชุดข้อสอบ × จำนวนข้อ)** — confidence interval คือราคาขั้นต่ำ
> ของคำว่า "ผลการทดลอง"

---

## 3. สมการ (Equation)

### 3.1 Wilson score interval — CI ประจำซีรีส์

$$\frac{\hat p + \frac{z^2}{2n}}{1 + \frac{z^2}{n}} \pm \frac{z}{1 + \frac{z^2}{n}}\sqrt{\frac{\hat p(1-\hat p)}{n} + \frac{z^2}{4n^2}}$$

ไม่ใช้ normal approximation เพราะมันพังตรงที่เราต้องใช้บ่อยที่สุด: ที่ $\hat p = 0$ หรือ $1$
มันให้ช่วงกว้าง**ศูนย์** — `wilson_ci(0, 30)` ของ kobeval ตอบ $[0\%, 11.4\%]$ ซึ่งซื่อสัตย์กว่า

### 3.2 Length-normalised multiple-choice scoring

$$\hat i = \arg\max_i \frac{\log p_\theta(c_i \mid q)}{|c_i|^{\gamma}}$$

- $\gamma = 0$ → log-prob รวมดิบ ๆ (ลำเอียงเข้าข้างตัวเลือกสั้น) — คือ `acc` ใน lm-eval
- $\gamma = 1$ → log-prob เฉลี่ยต่อ token — คือ `acc_norm`

สองบรรทัดนี้**ให้คะแนนไม่เท่ากันและบางครั้งสลับอันดับโมเดล** โดยที่ paper ส่วนใหญ่
ไม่บอกเลยว่าใช้ค่าไหน — หัวข้อ 7.1 จะสาธิตการสลับ argmax ด้วยตัวอย่างที่สร้างขึ้นเอง

### 3.3 pass@k แบบไม่ลำเอียง (Chen et al., 2021)

$$\widehat{\text{pass@}k} = 1 - \binom{n-c}{k}\Big/\binom{n}{k}$$

ตัวประมาณ plug-in $1-(1-c/n)^k$ ลำเอียงตาม Jensen's inequality — kobeval ใช้สูตรทวินามข้างบน

### 3.4 McNemar's test — เทียบสองโมเดลบนข้อสอบชุดเดียวกัน

$$\chi^2 = \frac{(|b-c|-1)^2}{b+c}$$

$b$ = ข้อที่ A ถูกแต่ B ผิด, $c$ = ข้อที่ B ถูกแต่ A ผิด — ข้อที่ทั้งคู่ถูกหรือทั้งคู่ผิด
**ไม่อยู่ในสูตรเลย** เพราะมันไม่ได้บอกว่าใครเก่งกว่า

### 3.5 Contamination check

$$\text{overlap}_k(x) = \frac{|G_k(x) \cap G_k(\mathcal{D})|}{|G_k(x)|}$$

$G_k(\cdot)$ = เซตของ character $k$-gram (ภาษาไทยตัดคำกำกวม จึงใช้ระดับตัวอักษร)
ค่า $k$ เองก็เป็นการตัดสินใจที่ต้องรายงาน — เราจะรันทั้ง $k=5$ (ไวมาก) และ $k=20$ (เข้มงวด)


---

## 5. เตรียมสภาพแวดล้อม (Environment) — และวัด baseline

หลักการของทั้งซีรีส์: **วัดก่อนเสมอ**

เซลล์ถัดไปคือ Cell 1 ซึ่งวัดผลโมเดลตั้งต้นด้วย KobEval-TH **ครบทั้ง 4 slice**
ตอนนี้เป็นตอนวัดผลโดยเฉพาะ เราจึงจ่ายงบเวลาให้การวัดเต็มที่ (ราว 9 นาที)
ต่างจากตอนเทรนที่ตัดเหลือ slice เดียวเพื่อเก็บงบไว้เทรน

สังเกตค่า `th_ratio` เป็นพิเศษ — มันคือสัดส่วนตัวอักษรไทยในคำตอบ
จุดที่โมเดลขนาดนี้พังบ่อยที่สุดกับคำถามภาษาไทย ไม่ใช่การตอบผิด
แต่คือการ **ตอบเป็นภาษาอังกฤษหรือภาษาจีนอย่างมั่นใจ** ซึ่ง accuracy อย่างเดียวมองไม่เห็น

โมเดลที่โหลดในเซลล์นี้ (`Qwen/Qwen3-0.6B`) จะรับสามบทบาทตลอดโน้ตบุ๊ก:
เป็น **anchor** ของตารางเปรียบเทียบ, เป็น **ผู้ถูกวัด** ในการทดลองแกว่งคะแนน
และเป็น **กรรมการ (judge)** ในหัวข้อ 7.3 — บทบาทสุดท้ายนี้มีปัญหาในตัวมันเอง
ซึ่งเราจะพูดตรง ๆ ตอนถึงหัวข้อนั้น


In [ ]:
# =============================================================================
# CELL 1 -- BASELINE (วัดผลก่อนแตะอะไรทั้งสิ้น)
# เซลล์นี้เป็นเซลล์โค้ดที่สองของทุกตอนในซีรีส์
# =============================================================================
import gc
import json
import math
import re
import time

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-0.6B"
BASE_ID = "Qwen/Qwen3-0.6B-Base"       # จะถูกใช้ในหัวข้อ 8 (กวาด checkpoint)
KOBEVAL_SLICES = ["TH-KNOW", "TH-MATH", "TH-INSTR", "TH-SAFE"]  # ครบ 4 slice
DEV = "cuda:0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,             # <-- ห้ามใช้ค่า auto เด็ดขาดบน T4 (ดูคอมเมนต์ Cell 0)
    attn_implementation=ATTN_IMPL, # <-- sdpa เท่านั้น
    device_map=DEV,
)
model.eval()

cfg = model.config
print(f"โหลดโมเดลแล้ว: {MODEL_ID}")
print(f"dtype จริงของ parameter: {next(model.parameters()).dtype}")
print(f"จำนวนพารามิเตอร์: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print("--- ค่าจริงจาก config ของ Qwen3-0.6B (ไม่ใช่ค่าที่จำมา) ---")
print(f"  num_hidden_layers   : {cfg.num_hidden_layers}")
print(f"  num_attention_heads : {cfg.num_attention_heads}")
print(f"  num_key_value_heads : {cfg.num_key_value_heads}")
print(f"  head_dim            : {getattr(cfg, 'head_dim', None)}")
print(f"  vocab_size          : {cfg.vocab_size}")

# รันเบนช์มาร์กกลาง -- สัญญาการวัดผลถูกตรึงไว้ใน kobeval แล้ว
# (greedy, max_new_tokens=256, enable_thinking=False, seed=42)
t0 = time.time()
baseline = evaluate(
    model,
    tokenizer,
    slices=KOBEVAL_SLICES,
    seed=42,
    model_name="Qwen3-0.6B (baseline)",
    out_path="results_baseline.json",
)
print(f"\nใช้เวลาวัด baseline: {time.time() - t0:.0f} วินาที")

for name, s in baseline["slices"].items():
    print(
        f"{name:9s} acc={s['accuracy']:.3f} "
        f"[95% CI {s['ci_low']:.3f}-{s['ci_high']:.3f}]  "
        f"th_ratio={s['th_ratio']:.2f}  len={s['mean_output_len']:.0f}"
    )

print(f"\nรวมทุก slice: acc={baseline['overall']['accuracy']:.3f}  "
      f"th_ratio={baseline['overall']['th_ratio']:.2f}")
print("\nสังเกต ci_low-ci_high ของแต่ละ slice: กว้างราว 15-18 จุดที่ n=30")
print("นี่คือเหตุผลที่ทั้งซีรีส์ห้ามอ่านความต่าง 1-2 ข้อว่าเป็นความก้าวหน้า")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 4 -- เห็นภาพสมการ 3.1: ความกว้างของ Wilson CI เป็นฟังก์ชันของ n
# เซลล์นี้ไม่ใช้ GPU และไม่ใช้เน็ต
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np

from kobeval import use_thai_font

use_thai_font()

# จุดที่ normal approximation โกหก: ที่ p=0 มันให้ช่วงกว้างศูนย์
lo, hi = wilson_ci(0, 30)
print(f"wilson_ci(0, 30)            = ({lo:.3f}, {hi:.3f})   <- ค่าจริงอาจสูงถึง 11.4%")
print(f"normal approximation ที่ p=0 = (0.000, 0.000)   <- มั่นใจ 100% จากแค่ 30 ตัวอย่าง")

ns = np.unique(np.logspace(1, 4, 120).astype(int))
fig, ax = plt.subplots(figsize=(8.5, 4.6))
for p, color in [(0.60, "#2563eb"), (0.75, "#f59e0b"), (0.90, "#dc2626")]:
    halfwidth = []
    for n in ns:
        s = int(round(p * n))
        w_lo, w_hi = wilson_ci(s, int(n))
        halfwidth.append(100 * (w_hi - w_lo) / 2)
    ax.plot(ns, halfwidth, lw=2, color=color, label=f"accuracy = {p:.2f}")

# จุดเน้นที่ n=100 -- ขนาดชุดข้อสอบไทยที่พบบ่อย
s100_lo, s100_hi = wilson_ci(75, 100)
ax.axvline(100, ls=":", color="#64748b", lw=1)
ax.annotate(f"n=100: บวกลบ {100 * (s100_hi - s100_lo) / 2:.1f} จุด (ที่ acc 75%)",
            (100, 100 * (s100_hi - s100_lo) / 2),
            textcoords="offset points", xytext=(10, 10), fontsize=9)

ax.set_xscale("log")
ax.set_xlabel("จำนวนข้อสอบ n (แกน log)")
ax.set_ylabel("ครึ่งความกว้างของ Wilson 95% CI (จุดเปอร์เซ็นต์)")
ax.set_title("ชุดทดสอบ 100 ข้อ แยกโมเดลที่ห่างกัน 4 จุดไม่ได้ ไม่ว่าจะรันซ้ำกี่รอบ",
             fontsize=11)
ax.legend()
ax.grid(alpha=0.3, which="both")
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig("fig_ci_width.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nบันทึกรูปแล้ว: fig_ci_width.png")
print("ความกว้างหดตาม 1/sqrt(n): อยากได้ CI แคบลง 10 เท่า ต้องจ่ายข้อสอบเพิ่ม 100 เท่า")


---

## 6. เตรียมข้อมูล (Data) — ด่านตรวจ license ที่ห้ามข้าม

ตอนนี้ใช้ข้อสอบ 3 ชุด:

1. **`scb10x/thai_exam`** — ข้อสอบปรนัยจากสนามสอบจริงของไทย (O-NET, IC, TGAT, TPAT-1, A-Level)
   คือ benchmark ภาษาไทยที่ leaderboard สาธารณะใช้กันมากที่สุด
2. **`VISAI-AI/gsm8k-thai`** — GSM8K ฉบับแปลไทย โจทย์เลขที่ต้อง**สร้างคำตอบเอง**
   ไม่มีตัวเลือกให้เทียบ log-likelihood จึงเป็นสนามทดสอบของโหมด generative โดยเฉพาะ
3. **KobEval-TH** — ชุดประจำซีรีส์ที่วัดไปแล้วใน Cell 1

> **ด่านตรวจ license — เซลล์ถัดไปบังคับขั้นตอนนี้**
> ข้อสอบจริงมีเจ้าของ ชุดข้อมูลที่ derive จากข้อสอบจริงมีเงื่อนไขการใช้ที่**ต้องอ่านเอง**
> เซลล์ถัดไปดึงช่อง `license` จาก dataset card ขึ้นมาพิมพ์ แล้วเทียบกับรายชื่อ license
> แบบเปิดที่เรายอมรับ ถ้าไม่ผ่าน **หัวข้อที่ใช้ชุดนั้นทั้งหมดจะถูกข้ามพร้อมข้อความอธิบาย**
> ไม่ใช่รันต่อแบบเงียบ ๆ — การวัดผลที่เริ่มจากการละเมิดเงื่อนไขข้อมูล ไม่มีทางเรียกว่า rigorous

ถ้าด่าน `thai_exam` ไม่ผ่านบนเครื่องคุณ (เช่น card ถูกแก้ license ภายหลัง)
โน้ตบุ๊กจะสร้างชุดปรนัยสำรองจาก TH-KNOW ของ KobEval-TH แทน (เฉลย + ตัวลวง 3 ตัว
ที่สุ่มจากเฉลยข้ออื่นแบบ deterministic) เพื่อให้ทุกเซลล์ถัดไปยังรันได้ครบ


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 6 -- ด่านตรวจ license แล้วจึงโหลดข้อสอบ
# -----------------------------------------------------------------------------
from datasets import load_dataset
from huggingface_hub import DatasetCard

N_EXAM = 60      # จำนวนข้อ thai_exam ที่ใช้ (ลดได้ถ้าเวลาไม่พอ)
N_GSM = 20       # จำนวนข้อ gsm8k-thai ที่ใช้
N_SHOT = 5       # จำนวนตัวอย่างใน few-shot prompt (หัวข้อ 9)

# license แบบเปิดที่เรายอมรับให้ "รันต่ออัตโนมัติ" ได้
PERMISSIVE = {"apache-2.0", "mit", "cc-by-4.0", "cc-by-sa-4.0", "cc0-1.0",
              "bsd-3-clause", "openrail"}


def license_gate(dataset_id):
    """พิมพ์ช่อง license จาก dataset card แล้วตัดสินว่ารันต่อได้ไหม

    คืน (ok, license_str) -- ok=False แปลว่าหัวข้อที่ใช้ชุดนี้ต้องหยุด
    """
    try:
        card = DatasetCard.load(dataset_id)
        lic = card.data.get("license")
        lic = lic[0] if isinstance(lic, list) and lic else lic
    except Exception as exc:
        print(f"[GATE] {dataset_id}: โหลด dataset card ไม่ได้ ({exc}) -> หยุดส่วนนี้")
        return False, None

    print(f"[GATE] {dataset_id}: license = {lic!r}")
    if isinstance(lic, str) and lic.lower() in PERMISSIVE:
        print(f"[GATE] {dataset_id}: เป็น license แบบเปิดที่รู้จัก -> ไปต่อได้")
        return True, lic

    print(f"[GATE] {dataset_id}: license ไม่อยู่ในรายชื่อแบบเปิดที่ยอมรับอัตโนมัติ")
    print("       หยุดส่วนที่ใช้ชุดนี้ทั้งหมด -- ไปอ่านเงื่อนไขบน dataset card ด้วยตาตัวเอง")
    print("       ก่อนตัดสินใจใช้ต่อ อย่าให้โค้ดตัดสินใจเรื่องกฎหมายแทนคุณ")
    return False, lic


EXAM_OK, EXAM_LICENSE = license_gate("typhoon-ai/thai_exam")
GSM_OK, GSM_LICENSE = license_gate("VISAI-AI/gsm8k-thai")

# --- thai_exam: ปรนัย 4-5 ตัวเลือก ------------------------------------------
LETTERS = ["a", "b", "c", "d", "e"]
EXAM_ITEMS, EXAM_FEWSHOT = [], []
EXAM_SOURCE = None

if EXAM_OK:
    exam_test = load_dataset("typhoon-ai/thai_exam", "onet", split="test")
    exam_train = load_dataset("typhoon-ai/thai_exam", "onet", split="train")
    print(f"thai_exam(onet) columns: {exam_test.column_names}  "
          f"test={len(exam_test)} train={len(exam_train)}")

    def to_item(row):
        choices = [(l, str(row[l]).strip()) for l in LETTERS
                   if isinstance(row.get(l), str) and str(row[l]).strip()]
        ans = str(row.get("answer", "")).strip().lower()
        letters = [l for l, _ in choices]
        if len(choices) < 3 or ans not in letters:
            return None
        return {
            "question": str(row["question"]).strip(),
            "choices": [c for _, c in choices],
            "letters": letters,
            "answer_idx": letters.index(ans),
        }

    for row in exam_test:
        item = to_item(row)
        if item and len(item["question"]) < 600:
            EXAM_ITEMS.append(item)
        if len(EXAM_ITEMS) >= N_EXAM:
            break
    for row in exam_train:
        item = to_item(row)
        if item and len(item["question"]) < 300:
            EXAM_FEWSHOT.append(item)
        if len(EXAM_FEWSHOT) >= N_SHOT:
            break
    EXAM_SOURCE = "typhoon-ai/thai_exam (onet/test)"

if not EXAM_ITEMS:
    # ชุดสำรอง: สร้างปรนัยจาก TH-KNOW (เฉลยจริง + ตัวลวง 3 ตัวจากเฉลยข้ออื่น)
    from kobeval import load_slice

    know = load_slice("TH-KNOW")
    golds = [it["answers"][0] for it in know]
    rng = random.Random(SEED)
    for i, it in enumerate(know):
        distractors = [g for j, g in enumerate(golds) if j != i]
        opts = [it["answers"][0]] + rng.sample(distractors, 3)
        order = list(range(4))
        rng.shuffle(order)
        EXAM_ITEMS.append({
            "question": it["prompt"],
            "choices": [opts[k] for k in order],
            "letters": LETTERS[:4],
            "answer_idx": order.index(0),
        })
    EXAM_FEWSHOT = EXAM_ITEMS[-N_SHOT:]
    EXAM_ITEMS = EXAM_ITEMS[:-N_SHOT]
    EXAM_SOURCE = "KobEval-TH TH-KNOW (แปลงเป็นปรนัย -- ชุดสำรองเมื่อด่าน license ไม่ผ่าน)"

print(f"\nชุดปรนัยที่ใช้: {EXAM_SOURCE}")
print(f"จำนวนข้อ: {len(EXAM_ITEMS)}  (few-shot pool: {len(EXAM_FEWSHOT)})")
print("ตัวอย่างข้อแรก:")
print("  คำถาม :", EXAM_ITEMS[0]["question"][:120].replace(chr(10), " "))
for l, c in zip(EXAM_ITEMS[0]["letters"], EXAM_ITEMS[0]["choices"]):
    print(f"    {l}. {c[:60]}")
print("  เฉลย  :", EXAM_ITEMS[0]["letters"][EXAM_ITEMS[0]["answer_idx"]])

# --- gsm8k-thai: generative -------------------------------------------------
GSM_ITEMS = []
if GSM_OK:
    gsm = load_dataset("VISAI-AI/gsm8k-thai", split="test")
    print(f"\ngsm8k-thai columns: {gsm.column_names}  n={len(gsm)}")
    for row in gsm:
        q = str(row["translated_question"]).strip()
        gold_text = str(row["translated_answer"])
        m = re.search(r"####\s*(-?[\d,]+)", gold_text)
        if not m or not (30 <= len(q) <= 400):
            continue
        GSM_ITEMS.append({"question": q, "gold": int(m.group(1).replace(",", ""))})
        if len(GSM_ITEMS) >= N_GSM:
            break
    print(f"gsm8k-thai ที่ใช้: {len(GSM_ITEMS)} ข้อ | ตัวอย่างเฉลย: {GSM_ITEMS[0]['gold']}")
else:
    print("\n[ข้าม] gsm8k-thai ไม่ผ่านด่าน license -- หัวข้อ generative/judge จะรายงานเฉพาะชุดที่เหลือ")


---

## 7. โค้ดหลัก (Main code)

### 7.1 โหมดที่ 1 — log-likelihood multiple-choice พร้อมปุ่ม $\gamma$

ไม่มีการ generate เลยสักตัวอักษร: ให้โมเดลอ่าน `คำถาม + ตัวเลือก` ทีละตัวเลือก
แล้วรวม log-prob เฉพาะ token ของตัวเลือก จากนั้นหารด้วย $|c_i|^\gamma$ ตามสมการ 3.2

เราแยกฟังก์ชันเป็นสองชั้นโดยตั้งใจ:

- `choice_scores()` — ส่วนที่ต้องใช้โมเดล คืน `(ผลรวม log-prob, จำนวน token)` ต่อตัวเลือก
- `pick()` — ส่วนที่เป็นคณิตศาสตร์ล้วน รับ list ของคู่ตัวเลขแล้วเลือกตาม $\gamma$

การแยกแบบนี้ทำให้เรา **ทดสอบเรื่อง $\gamma$ ได้โดยไม่ต้องมี GPU**:
ท้ายเซลล์มีตัวอย่าง 2 ตัวเลือกที่สร้างขึ้นเองซึ่ง $\gamma=0$ กับ $\gamma=1$
เลือกคนละคำตอบ — นี่คือ `acc` vs `acc_norm` ใน lm-evaluation-harness แบบจับต้องได้

หมายเหตุความยุติธรรม (กับดักข้อ 3 ของบทความ): โหมดนี้ใช้ prompt แบบ completion ดิบ
**เหมือนกันทุกโมเดล** ไม่ใส่ chat template ให้ใครเป็นพิเศษ เพราะ roster ของเรามีทั้ง
base และ instruct — เซลล์จะพิมพ์ prompt จริงของข้อแรกให้ตรวจด้วยตา


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 7.1 -- log-likelihood MC scorer (สมการ 3.2) + self-check เรื่อง gamma
# -----------------------------------------------------------------------------
def mc_prompt(item, fewshot=()):
    """prompt แบบ completion ดิบ ใช้เหมือนกันทุกโมเดล (base และ instruct)"""
    parts = []
    for ex in fewshot:
        parts.append(f"คำถาม: {ex['question']}\nคำตอบ: {ex['choices'][ex['answer_idx']]}\n")
    parts.append(f"คำถาม: {item['question']}\nคำตอบ: ")
    return "\n".join(parts)


@torch.no_grad()
def choice_scores(m, tok, item, fewshot=()):
    """คืน [(sum_logp, n_tokens), ...] หนึ่งคู่ต่อหนึ่งตัวเลือก -- ยังไม่ตัดสินอะไร"""
    prompt = mc_prompt(item, fewshot)
    q_ids = tok(prompt, return_tensors="pt").input_ids
    out = []
    for c in item["choices"]:
        full_ids = tok(prompt + c, return_tensors="pt").input_ids.to(m.device)
        logits = m(full_ids).logits[:, :-1]
        targets = full_ids[:, 1:]
        logp = torch.log_softmax(logits.float(), -1).gather(
            -1, targets.unsqueeze(-1)).squeeze(-1)
        ans = logp[0, q_ids.shape[1] - 1:]          # เฉพาะ token ของตัวเลือก
        out.append((float(ans.sum()), int(ans.numel())))
    return out


def pick(scores, gamma):
    """สมการ 3.2: argmax ของ sum_logp / n_tokens**gamma -- คณิตศาสตร์ล้วน ไม่แตะโมเดล"""
    vals = [s / (n ** gamma) for s, n in scores]
    return max(range(len(vals)), key=lambda i: vals[i])


# --- self-check ที่ไม่ต้องใช้ GPU: gamma พลิก argmax ได้จริง ---------------------
# ตัวเลือก A: สั้น 2 token, log-prob รวม -6.0  (เฉลี่ย -3.0 ต่อ token -- โมเดลไม่ค่อยมั่นใจ)
# ตัวเลือก B: ยาว 8 token, log-prob รวม -8.0  (เฉลี่ย -1.0 ต่อ token -- มั่นใจกว่ามาก)
_toy = [(-6.0, 2), (-8.0, 8)]
_pick_g0 = pick(_toy, gamma=0.0)     # เทียบผลรวมดิบ: -6.0 > -8.0 -> เลือก A
_pick_g1 = pick(_toy, gamma=1.0)     # เทียบเฉลี่ยต่อ token: -3.0 < -1.0 -> เลือก B
print(f"self-check | gamma=0 เลือกตัวเลือก index {_pick_g0} (ตัวสั้น)")
print(f"self-check | gamma=1 เลือกตัวเลือก index {_pick_g1} (ตัวยาวที่มั่นใจกว่า)")
assert _pick_g0 == 0 and _pick_g1 == 1, "gamma ต้องพลิก argmax ของตัวอย่างนี้ได้"
print("self-check | ผ่าน: การตั้งค่า gamma ตัวเดียวเปลี่ยนคำตอบของ scorer ได้จริง")
print("            (นี่คือ acc vs acc_norm ใน lm-evaluation-harness)")

# --- รันจริงกับ anchor model: ให้คะแนนครั้งเดียว ตัดสินสองแบบ ---------------------
print(f"\nprompt จริงของข้อแรก (ใช้เหมือนกันทุกโมเดล -- ตรวจความยุติธรรมด้วยตา):")
print("-" * 60)
print(mc_prompt(EXAM_ITEMS[0])[:400])
print("-" * 60)

t0 = time.time()
ANCHOR_SCORES = [choice_scores(model, tokenizer, it) for it in EXAM_ITEMS]
print(f"ให้คะแนน {len(EXAM_ITEMS)} ข้อ ใช้เวลา {time.time() - t0:.0f} วินาที")

anchor_g1 = [pick(s, 1.0) == it["answer_idx"] for s, it in zip(ANCHOR_SCORES, EXAM_ITEMS)]
anchor_g0 = [pick(s, 0.0) == it["answer_idx"] for s, it in zip(ANCHOR_SCORES, EXAM_ITEMS)]

for tag, rec in [("gamma=1 (acc_norm)", anchor_g1), ("gamma=0 (acc)     ", anchor_g0)]:
    k, n = sum(rec), len(rec)
    lo, hi = wilson_ci(k, n)
    print(f"{MODEL_ID} | loglik {tag}: {k}/{n} = {k / n:.3f} [CI {lo:.3f}-{hi:.3f}]")

n_flip = sum(1 for a, b in zip(anchor_g1, anchor_g0) if a != b)
print(f"\nข้อที่สองแบบตัดสินไม่ตรงกัน: {n_flip}/{len(EXAM_ITEMS)} ข้อ")
print("โมเดลตัวเดิม น้ำหนักเดิมทุกไบต์ -- ต่างกันแค่ gamma ที่ paper ส่วนใหญ่ไม่รายงาน")


### 7.2 โหมดที่ 2 — generative exact-match พร้อม normalization ภาษาไทย

"๕๐" กับ "50" กับ " 50 " คือคำตอบเดียวกัน แต่ `==` ของ Python ไม่คิดแบบนั้น
ด่านที่ exact-match ภาษาไทยตายบ่อยที่สุดคือเลขไทย ๐-๙ กับช่องว่าง

เราเขียน `normalize_thai()` จากศูนย์ให้เห็นไส้ใน แล้ว **unit test มันก่อนใช้**
เพราะบั๊กในตัวตรวจคือ contamination กลับด้าน: มันทำให้โมเดลดูแย่กว่าจริง
อย่างเป็นระบบ และไม่มี error ใด ๆ ฟ้อง — ร้ายกว่านั้น มันกระจายไม่สม่ำเสมอ:
โมเดลที่ CPT ด้วยเอกสารราชการ (ซึ่งใช้เลขไทยเยอะ) จะโดนหักคะแนนมากกว่าเพื่อน

ส่วนการดึง "เลขคำตอบสุดท้าย" ออกจาก chain-of-thought เราใช้
`kobeval.extract_final_int` ตัวเดียวกับที่ grader ของ TH-MATH ใช้มาทั้งซีรีส์
— การตัดสินใจชั้นนี้ก็กระทบคะแนนและต้องรายงานเช่นกัน


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 7.2 -- normalize_thai จากศูนย์ + unit test + generative exact-match
# -----------------------------------------------------------------------------
from kobeval import extract_final_int, thai_digits_to_arabic

THAI_DIGITS = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")


def normalize_thai(s):
    """normalize คำตอบสั้นภาษาไทยก่อนเทียบ exact-match"""
    s = s.strip().translate(THAI_DIGITS)   # เลขไทย ๐-๙ -> อารบิก
    s = s.replace(",", "")                 # 1,000 -> 1000
    s = re.sub(r"\s+", "", s)              # ไทยไม่ใช้ช่องว่างแบ่งคำ -- ตัดทิ้งทั้งหมด
    return s


# --- unit test ของตัวตรวจ (รันได้บน CPU ไม่ต้องมีโมเดล) --------------------------
_tricky = [
    ("๕๐ บาท", "50 บาท", True),            # เลขไทย vs อารบิก
    ("  ๑,๒๓๔  ", "1234", True),            # เลขไทย + คั่นหลักพัน + ช่องว่างหัวท้าย
    ("๒๕๖๗", "2567", True),                 # ปี พ.ศ. ล้วน
    ("กรุงเทพ มหานคร", "กรุงเทพมหานคร", True),  # ช่องว่างกลางคำ (โมเดลชอบแทรก)
    ("๕๑ บาท", "50 บาท", False),            # ต้องไม่ใจดีเกิน: 51 != 50
]
for a, b, want in _tricky:
    got = normalize_thai(a) == normalize_thai(b)
    status = "ok " if got == want else "FAIL"
    print(f"  {status} normalize_thai({a!r}) == normalize_thai({b!r}) -> {got}")
    assert got == want, f"normalizer พังที่คู่ {a!r} vs {b!r}"
print("unit test ของ normalize_thai ผ่านทั้งหมด")

# kobeval มีตัวช่วยชุดเดียวกันที่ผ่าน test suite ของ repo มาแล้ว -- เทียบกันตรง ๆ
assert thai_digits_to_arabic("พ.ศ. ๒๕๖๗") == "พ.ศ. 2567"
assert extract_final_int("คิดเลขยาว ๆ 12+8=20 ดังนั้นคำตอบคือ 20 บาท") == 20
print("cross-check กับ kobeval.thai_digits_to_arabic / extract_final_int ผ่าน")


def chat_ids(question, thinking=False):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        add_generation_prompt=True,
        enable_thinking=thinking,          # สัญญาประจำซีรีส์: ปิด เว้นแต่ประกาศ
        return_tensors="pt",
    ).to(DEV)


@torch.no_grad()
def generate_answer(m, question, max_new=192, thinking=False):
    torch.manual_seed(SEED)                # re-seed ต่อข้อ ตามสัญญาของ kobeval
    ids = chat_ids(question, thinking)
    out = m.generate(ids, max_new_tokens=max_new, do_sample=False,
                     pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
    if "</think>" in text:                 # กัน thinking block รั่วมาปนคำตอบ
        text = text.split("</think>", 1)[1]
    return text.strip()


GSM_RECORDS = []
if GSM_ITEMS:
    t0 = time.time()
    for i, it in enumerate(GSM_ITEMS, 1):
        pred_text = generate_answer(
            model, it["question"] + "\nแสดงวิธีคิดสั้น ๆ แล้วจบด้วยบรรทัด: คำตอบคือ <ตัวเลข>")
        pred_int = extract_final_int(thai_digits_to_arabic(pred_text))
        GSM_RECORDS.append({
            "question": it["question"],
            "gold": it["gold"],
            "pred_text": pred_text,
            "pred_int": pred_int,
            "correct": pred_int == it["gold"],
            "th_ratio": th_ratio(pred_text),
        })
        if i % 5 == 0:
            print(f"  gsm8k-thai: {i}/{len(GSM_ITEMS)} ({time.time() - t0:.0f}s)")

    k = sum(r["correct"] for r in GSM_RECORDS)
    n = len(GSM_RECORDS)
    lo, hi = wilson_ci(k, n)
    mean_th = sum(r["th_ratio"] for r in GSM_RECORDS) / n
    print(f"\nGSM8K-TH generative exact-match: {k}/{n} = {k / n:.3f} [CI {lo:.3f}-{hi:.3f}]")
    print(f"th_ratio เฉลี่ยของคำตอบ: {mean_th:.2f}  <- ต่ำ = โมเดลหนีไปคิดเป็นภาษาอื่น")
    print("\nตัวอย่างคำตอบข้อแรก:")
    print(" ", GSM_RECORDS[0]["pred_text"][:200].replace(chr(10), " "))
    print(f"  ดึงคำตอบได้: {GSM_RECORDS[0]['pred_int']}  เฉลย: {GSM_RECORDS[0]['gold']}")
else:
    print("[ข้าม] ไม่มีข้อมูล gsm8k-thai (ดูด่าน license ในหัวข้อ 6)")


### 7.3 โหมดที่ 3 — LLM-as-judge พร้อม rubric เป็นลายลักษณ์อักษร

โหมดสุดท้ายให้ LLM อีกตัวตรวจคำตอบตาม rubric ภาษาไทยที่เขียนไว้ชัด ๆ
ของจริงควรใช้ judge ที่**แข็งแรงกว่าผู้ถูกวัดมาก** (เช่น endpoint ของโมเดลใหญ่)
แต่บน Colab ฟรีเราไม่มี endpoint ให้เรียก จึงใช้โมเดลตัวเดียวกับผู้ถูกวัด
เป็น judge — **ซึ่งผิดหลักการโดยตั้งใจ** เพื่อให้เห็นข้อจำกัดสองข้อกับตา:

1. **Self-enhancement bias** — งานวิจัยหลายชิ้นพบว่า LLM judge ให้คะแนนคำตอบ
   สไตล์ตระกูลเดียวกับตัวเองสูงกว่าที่ควร ยิ่ง judge กับผู้ถูกวัดคือ**ตัวเดียวกัน**
   ตัวเลขยิ่งหอมหวานผิดปกติ
2. **Judge ที่อ่อนแอตรวจเลขไม่เป็น** — โจทย์ GSM8K ต้องคิดเลขเพื่อตรวจ
   ถ้า judge คิดเลขผิดเอง คะแนนที่ให้ก็มั่ว

ทางแก้ที่ถูกไม่ใช่หา judge ที่ "เป็นกลาง" (ไม่มีอยู่จริง) แต่คือ **รายงานชื่อ judge
และ rubric เสมอ** — เซลล์ถัดไปจึงบันทึกทั้งสองอย่างลง `results.json`
ผลจาก judge ที่ไม่บอกว่า judge คือใคร ก็เป็นข่าวลืออีกชนิดหนึ่ง


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 7.3 -- LLM-as-judge (ใช้โมเดลท้องถิ่นเป็นกรรมการ + พิมพ์คำเตือน bias)
# -----------------------------------------------------------------------------
JUDGE_MODEL_NAME = MODEL_ID + " (ตัวเดียวกับผู้ถูกวัด!)"

JUDGE_RUBRIC = """คุณคือกรรมการตรวจข้อสอบ ตัดสินตามเกณฑ์นี้เท่านั้น:
1 = ใจความถูกต้องตรงกับเฉลย (ยอมรับการสะกดต่างกัน เลขไทย/อารบิก
    และการเรียบเรียงคนละแบบที่ความหมายเดียวกัน)
0 = ผิด ตอบไม่ตรงคำถาม หรือไม่ตอบ
ห้ามให้คะแนนความสวยงามของภาษา ห้ามให้คะแนนความยาว
ตอบเป็น JSON เท่านั้น: {"score": 0 หรือ 1, "reason": "สั้น ๆ"}

โจทย์: {q}
เฉลย: {gold}
คำตอบของโมเดล: {pred}"""

print("=" * 70)
print("คำเตือน SELF-ENHANCEMENT BIAS -- อ่านก่อนเชื่อตัวเลขของหัวข้อนี้")
print(f"  judge คือ {JUDGE_MODEL_NAME}")
print("  กรรมการกับผู้เข้าสอบเป็นโมเดลตัวเดียวกัน คะแนนจึงมีแนวโน้มลำเอียงเข้าข้าง")
print("  สไตล์ของตัวเอง และ judge ขนาด 0.6B ตรวจเลขผิดเองได้บ่อย")
print("  ตัวเลขนี้มีไว้สาธิตกลไก ไม่ใช่คะแนนที่ควรอ้างอิง -- ในงานจริงให้เสียบ")
print("  endpoint ของโมเดลที่แข็งแรงกว่ามาก และรายงานชื่อ judge เสมอ")
print("=" * 70)


@torch.no_grad()
def judge_one(question, gold, pred, max_new=96):
    prompt = JUDGE_RUBRIC.replace("{q}", question[:400]) \
                         .replace("{gold}", str(gold)) \
                         .replace("{pred}", pred[:400])
    verdict = generate_answer(model, prompt, max_new=max_new)
    m = re.search(r'"score"\s*:\s*([01])', verdict)
    if m is None:
        m = re.search(r"\b([01])\b", verdict)     # judge เล็ก ๆ ชอบตอบนอกรูปแบบ
    return (int(m.group(1)) if m else 0), verdict


JUDGE_RECORDS = []
if GSM_RECORDS:
    t0 = time.time()
    for i, r in enumerate(GSM_RECORDS, 1):
        score, verdict = judge_one(r["question"], r["gold"], r["pred_text"])
        JUDGE_RECORDS.append({"judge_score": score, "verdict": verdict[:200],
                              "em_correct": r["correct"]})
        if i % 5 == 0:
            print(f"  judge: {i}/{len(GSM_RECORDS)} ({time.time() - t0:.0f}s)")

    k = sum(j["judge_score"] for j in JUDGE_RECORDS)
    n = len(JUDGE_RECORDS)
    lo, hi = wilson_ci(k, n)
    agree = sum(1 for j in JUDGE_RECORDS if bool(j["judge_score"]) == j["em_correct"])
    print(f"\nGSM8K-TH ตาม judge: {k}/{n} = {k / n:.3f} [CI {lo:.3f}-{hi:.3f}]")
    print(f"judge เห็นตรงกับ exact-match: {agree}/{n} ข้อ")
    disagree = [i for i, j in enumerate(JUDGE_RECORDS) if bool(j["judge_score"]) != j["em_correct"]]
    if disagree:
        i = disagree[0]
        print(f"\nตัวอย่างข้อที่สองกรรมการเห็นไม่ตรงกัน (ข้อ {i}):")
        print(f"  exact-match บอก {GSM_RECORDS[i]['correct']}, judge บอก {JUDGE_RECORDS[i]['judge_score']}")
        print(f"  คำตัดสินของ judge: {JUDGE_RECORDS[i]['verdict'][:150]}")
else:
    print("[ข้าม] ไม่มีคำตอบจากหัวข้อ 7.2 ให้ตรวจ")


### 7.4 สิ่งที่มืออาชีพใช้ — lm-evaluation-harness

สามโหมดข้างบนเขียนเองเพื่อให้เห็นไส้ใน แต่งานจริงควรยืนบนเครื่องมือที่ community
ตรวจสอบแล้ว เซลล์ถัดไป**ตั้งใจคอมเมนต์ไว้ทั้งเซลล์ ไม่รันจริง** เพราะ lm-eval
ติดตั้ง dependency ก้อนใหญ่ทับสภาพแวดล้อมที่เรา pin ไว้ (และกินเวลาเกินงบ T4 ฟรี)
— เอาไว้รันบนเครื่องของคุณเองนอก Colab

บรรทัดที่สำคัญที่สุดคือ `--log_samples`: มันบันทึกคำตอบรายข้อ ทำให้
(1) คำนวณ Wilson CI เองได้ (2) จับคู่รายข้อทำ McNemar ได้ (3) ไล่ดูว่าข้อไหนผิดเพราะอะไร
และสังเกตว่า harness รายงานทั้ง `acc` และ `acc_norm` — คือ $\gamma=0$ กับ length-normalised
ของสมการ 3.2 ที่เราเพิ่งเขียนเองนั่นเอง


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 7.4 -- lm-evaluation-harness CLI (ตั้งใจคอมเมนต์ไว้ ไม่รันบน Colab ฟรี)
# -----------------------------------------------------------------------------
# pip install lm-eval
#
# # ดูชื่อ task ที่มีจริงในเวอร์ชันของคุณก่อน -- ชื่อ task เปลี่ยนข้ามเวอร์ชันบ่อย
# lm_eval --tasks list | grep -i thai
#
# lm_eval --model hf \
#     --model_args pretrained=Qwen/Qwen3-0.6B,dtype=float16 \
#     --tasks thai_exam \
#     --num_fewshot 5 \
#     --batch_size 8 \
#     --seed 42 \
#     --log_samples --output_path results/harness
#
# สิ่งที่ต้องจดลงรายงานเสมอเมื่อใช้ harness:
#   - เวอร์ชันของ lm-eval (คะแนนเทียบข้ามเวอร์ชันไม่ได้)
#   - acc หรือ acc_norm (gamma=0 หรือ length-normalised -- ดูหัวข้อ 7.1)
#   - จำนวน shot และ template
#   - dtype (float16 บน T4 -- ห้าม auto ด้วยเหตุผลเดียวกับ Cell 0)
print("เซลล์นี้ตั้งใจเป็นคอมเมนต์ทั้งเซลล์ -- อ่านแล้วไปรันนอก Colab")


---

## 8. ผลลัพธ์ (Results) — ทุก checkpoint บนแกนเดียว พร้อม error bar

### ธรรมเนียม `checkpoints/` ของซีรีส์

ตอนที่ 1–8 แต่ละตอนจบด้วยการเซฟผลงานของตัวเอง (LoRA adapter หรือน้ำหนักเต็ม)
ถ้าคุณรันตอนเหล่านั้นใน session เดียวกัน หรือ mount Google Drive แล้ว copy มาไว้
ตาม path ข้างล่าง โน้ตบุ๊กนี้จะหยิบขึ้นมาวัดให้อัตโนมัติ:

| checkpoint | path ที่มองหา | ชนิด |
|---|---|---|
| ตอนที่ 1 — CPT | `checkpoints/01_cpt` | น้ำหนักเต็ม |
| ตอนที่ 2 — SFT | `checkpoints/02_sft_lora` หรือ `qwen3-0.6b-th-sft-lora` | LoRA adapter |
| ตอนที่ 4 — DPO | `checkpoints/04_dpo_lora` หรือ `qwen3-0.6b-th-dpo-lora` | LoRA adapter |

checkpoint ที่หาไม่เจอจะถูก**ข้ามพร้อมข้อความบอก ไม่ใช่ error** — และตารางยังมี
อย่างน้อย 2 แถวเสมอ เพราะ `Qwen/Qwen3-0.6B` (instruct) กับ `Qwen/Qwen3-0.6B-Base`
โหลดจาก Hugging Face ได้ตรง ๆ ไม่ต้องพึ่งไฟล์ในเครื่อง

ทุกแถววัดด้วย **log-likelihood γ=1, 0-shot, prompt เดียวกันทุกตัวอักษร**
(โหมดที่ยุติธรรมกับ base model ที่สุด เพราะไม่ต้อง generate เป็นประโยค)
แถบ error คือ Wilson 95% CI จาก `kobeval.wilson_ci` — รูปนี้คือรูปหัวเรื่องของตอน


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 8.1 -- กวาดทุก checkpoint ที่หาเจอ ด้วย scorer เดียวกันทุกตัวอักษร
# -----------------------------------------------------------------------------
import os

from peft import PeftModel

ROSTER = [
    # (ป้ายชื่อ, HF id หรือ None, path ในเครื่องที่ลองตามลำดับ, ฐานของ adapter)
    ("Qwen3-0.6B-Base", BASE_ID, [], None),
    ("Qwen3-0.6B", MODEL_ID, [], None),
    ("01-CPT (full)", None, ["checkpoints/01_cpt"], None),
    ("02-SFT (LoRA)", None, ["checkpoints/02_sft_lora", "qwen3-0.6b-th-sft-lora"], MODEL_ID),
    ("04-DPO (LoRA)", None, ["checkpoints/04_dpo_lora", "qwen3-0.6b-th-dpo-lora"], MODEL_ID),
]

SWEEP = {}          # label -> {"correct": [bool ต่อข้อ], "accuracy", "ci_low", ...}


def eval_loglik(m, label):
    t0 = time.time()
    rec = [pick(choice_scores(m, tokenizer, it), 1.0) == it["answer_idx"]
           for it in EXAM_ITEMS]
    k, n = sum(rec), len(rec)
    lo, hi = wilson_ci(k, n)
    print(f"  {label:18s} {k:2d}/{n} = {k / n:.3f} [CI {lo:.3f}-{hi:.3f}] "
          f"({time.time() - t0:.0f}s)")
    return {"correct": rec, "n_correct": k, "n": n,
            "accuracy": k / n, "ci_low": lo, "ci_high": hi}


print(f"ข้อสอบ: {EXAM_SOURCE} | {len(EXAM_ITEMS)} ข้อ | loglik gamma=1, 0-shot\n")

# แถวของ anchor ใช้คะแนนที่คิดไว้แล้วในหัวข้อ 7.1 -- ไม่ต้องรันซ้ำ
k, n = sum(anchor_g1), len(anchor_g1)
lo, hi = wilson_ci(k, n)
SWEEP["Qwen3-0.6B"] = {"correct": list(anchor_g1), "n_correct": k, "n": n,
                       "accuracy": k / n, "ci_low": lo, "ci_high": hi}
print(f"  {'Qwen3-0.6B':18s} {k:2d}/{n} = {k / n:.3f} [CI {lo:.3f}-{hi:.3f}] (จากหัวข้อ 7.1)")

for label, hf_id, local_paths, adapter_base in ROSTER:
    if label in SWEEP:
        continue
    local = next((p for p in local_paths if os.path.isdir(p)), None)
    if hf_id is None and local is None:
        print(f"  {label:18s} -- ไม่พบไฟล์ใน {local_paths} -> ข้าม (ไม่ใช่ error)")
        continue
    try:
        if adapter_base is not None:            # LoRA adapter บนฐาน instruct
            m = AutoModelForCausalLM.from_pretrained(
                adapter_base,
                torch_dtype=DTYPE,
                attn_implementation=ATTN_IMPL,
                device_map=DEV,
            )
            m = PeftModel.from_pretrained(m, local)
        else:                                   # น้ำหนักเต็ม (HF id หรือ dir ในเครื่อง)
            m = AutoModelForCausalLM.from_pretrained(
                hf_id or local,
                torch_dtype=DTYPE,
                attn_implementation=ATTN_IMPL,
                device_map=DEV,
            )
        m.eval()
        SWEEP[label] = eval_loglik(m, label)
        # เพิ่ม KobEval-TH ต่อ checkpoint เพื่อเติมคอลัมน์ KobEval + th_ratio ในตารางหัวข้อ 8
        try:
            from kobeval import evaluate as _evaluate
            _kob = _evaluate(m, tokenizer, slices=["TH-KNOW", "TH-MATH", "TH-INSTR"],
                             seed=42, model_name=label, out_path=None,
                             compute_ppl=False, progress=False)
            SWEEP[label]["kobeval_acc"] = _kob["overall"]["accuracy"]
            SWEEP[label]["kobeval_th_ratio"] = _kob["overall"]["th_ratio"]
            print(f"    KobEval-TH: acc={_kob['overall']['accuracy']:.3f} "
                  f"th_ratio={_kob['overall']['th_ratio']:.2f}")
        except Exception as _e:
            print(f"    (KobEval ข้ามสำหรับ {label}: {str(_e)[:70]})")
        del m
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as exc:
        print(f"  {label:18s} -- โหลด/วัดไม่สำเร็จ ({exc}) -> ข้าม")

assert len(SWEEP) >= 2, "ต้องมีอย่างน้อย 2 แถวเสมอ (Base กับ Instruct มาจาก HF ตรง ๆ)"
print(f"\nวัดได้ {len(SWEEP)} checkpoint: {list(SWEEP)}")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 8.2 -- รูปหัวเรื่องของตอน: ทุก checkpoint พร้อม Wilson 95% CI
# -----------------------------------------------------------------------------
use_thai_font()

labels = list(SWEEP)
accs = np.array([100 * SWEEP[l]["accuracy"] for l in labels])
err_lo = np.array([100 * (SWEEP[l]["accuracy"] - SWEEP[l]["ci_low"]) for l in labels])
err_hi = np.array([100 * (SWEEP[l]["ci_high"] - SWEEP[l]["accuracy"]) for l in labels])
n_items = SWEEP[labels[0]]["n"]

fig, ax = plt.subplots(figsize=(1.9 * len(labels) + 3.5, 5.0))
bars = ax.bar(labels, accs, yerr=[err_lo, err_hi], capsize=5, color="#2563eb")
chance = 100.0 / len(EXAM_ITEMS[0]["choices"])
ax.axhline(chance, ls=":", color="#dc2626", lw=1.5)
ax.text(len(labels) - 0.5, chance + 1.5, f"เดาสุ่ม ~{chance:.0f}%",
        color="#dc2626", fontsize=9, ha="right")
for xi, l in enumerate(labels):
    ax.text(xi, accs[xi] + err_hi[xi] + 2,
            f"{SWEEP[l]['n_correct']}/{SWEEP[l]['n']}", ha="center", fontsize=9)
ax.set_ylabel("accuracy (%)")
ax.set_ylim(0, 105)
ax.set_title(f"ทุก checkpoint ของซีรีส์บนข้อสอบเดียวกัน ({n_items} ข้อ)\n"
             f"error bar = Wilson 95% CI -- ช่วงที่ซ้อนทับกันคือ 'แยกไม่ออก'",
             fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
plt.setp(ax.get_xticklabels(), rotation=12, ha="right")
fig.tight_layout()
fig.savefig("fig_all_checkpoints_ci.png", dpi=150, bbox_inches="tight")
plt.show()

print("บันทึกรูปแล้ว: fig_all_checkpoints_ci.png  <- รูปหัวเรื่องของตอนนี้")
print("วิธีอ่าน: ถ้าแถบ CI สองแท่งซ้อนทับกันมาก ข้อมูลยังไม่พอจะประกาศผู้ชนะ")
print("ไม่ว่าตัวเลขหน้าแท่งจะห่างกันกี่จุดก็ตาม")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 8.3 -- McNemar ระหว่างคู่ checkpoint ที่คะแนนใกล้กันที่สุด
# -----------------------------------------------------------------------------
from itertools import combinations

from kobeval import mcnemar

# --- self-check กับสูตรปิดก่อนใช้: b=9, c=1 (ตัวอย่างเดียวกับรูป 9.4 ในบทความ) ----
res = mcnemar(9, 1)
stat_closed = (abs(9 - 1) - 1) ** 2 / (9 + 1)                  # (|b-c|-1)^2/(b+c) = 4.9
p_closed = math.erfc(math.sqrt(stat_closed / 2.0))             # survival ของ chi2(1)
print(f"self-check | mcnemar(9,1): statistic={res['statistic']:.4f}  p={res['p_value']:.4f}")
print(f"self-check | สูตรปิด      : statistic={stat_closed:.4f}  p={p_closed:.4f}")
assert abs(res["statistic"] - stat_closed) < 1e-12
assert abs(res["p_value"] - p_closed) < 1e-12
assert abs(stat_closed - 4.9) < 1e-12, "ตัวอย่างในบทความ: chi2 ต้องเท่ากับ 4.90 พอดี"
print("self-check | ตรงกับสูตรปิดทุกหลัก -> ผ่าน (สองโมเดลห่าง 8 จุด แต่ p แค่ 0.027)")

# --- หา 'คู่ที่ใกล้กันที่สุด' แล้วถามว่าข้อมูลแยกมันออกจากกันได้ไหม ---------------------
pairs = list(combinations(SWEEP, 2))
closest = min(pairs, key=lambda p: abs(SWEEP[p[0]]["accuracy"] - SWEEP[p[1]]["accuracy"]))
A, B = closest
rec_a, rec_b = SWEEP[A]["correct"], SWEEP[B]["correct"]

a = sum(1 for x, y in zip(rec_a, rec_b) if x and y)          # ถูกทั้งคู่
b = sum(1 for x, y in zip(rec_a, rec_b) if x and not y)      # A ถูก B ผิด
c = sum(1 for x, y in zip(rec_a, rec_b) if not x and y)      # A ผิด B ถูก
d = sum(1 for x, y in zip(rec_a, rec_b) if not x and not y)  # ผิดทั้งคู่

print(f"\nคู่ที่คะแนนใกล้กันที่สุด: {A} ({SWEEP[A]['accuracy']:.3f}) "
      f"vs {B} ({SWEEP[B]['accuracy']:.3f})")
print("ตาราง contingency 2x2 (ตัวเลขจริง ไม่ใช่ตัวอย่าง):")
print(f"                    {B} ถูก   {B} ผิด")
print(f"  {A} ถูก        a={a:3d}      b={b:3d}")
print(f"  {A} ผิด        c={c:3d}      d={d:3d}")
print(f"  ข้อที่มีสาระต่อการเทียบ: b+c = {b + c} จาก {len(rec_a)} ข้อ "
      f"(a กับ d ไม่อยู่ในสูตรเลย)")

test = mcnemar(b, c)
warn = "  [n_discordant<25: ค่า p เป็นเพียงตัวชี้แนว]" if test["exact_recommended"] else ""
print(f"\nMcNemar: chi2={test['statistic']:.3f}  p={test['p_value']:.4f}  "
      f"-> {test['direction']}{warn}")
if not test["significant"]:
    print("อ่านผล: ข้อมูลเท่านี้ยังแยกสองโมเดลนี้ไม่ออก -- และการพูดแบบนี้ตรง ๆ")
    print("คือคำตอบที่ซื่อสัตย์ ไม่ใช่ความล้มเหลวของการทดลอง")


---

## 9. เปรียบเทียบ (Comparison) — โมเดลตัวเดิม วิธีวัดหลายแบบ

ทุกตอนก่อนหน้าเทียบ "โมเดลหลายตัว วิธีวัดเดียว" — หัวข้อนี้พลิกกลับ:
**โมเดลตัวเดียว (anchor `Qwen/Qwen3-0.6B` น้ำหนักเดิมทุกไบต์) วิธีวัดหลายแบบ**
แล้วดูว่าคะแนนแกว่งแค่ไหนจากการตัดสินใจที่ปกติไม่มีใครเขียนรายงาน:

| แถว | ต่างจากสัญญาประจำซีรีส์ตรงไหน |
|---|---|
| loglik $\gamma=1$, 0-shot | ไม่ต่าง — นี่คือสัญญา |
| loglik $\gamma=0$ | เปลี่ยนสูตร normalize ความยาว (สมการ 3.2) |
| loglik $\gamma=1$, 5-shot | เพิ่มตัวอย่างใน prompt |
| generative exact-match | เปลี่ยนโหมดวัดทั้งโหมด |
| LLM-as-judge | เปลี่ยนกรรมการ (ดูคำเตือนหัวข้อ 7.3) |
| `enable_thinking=True` | เปิดโหมดคิดในใจของ Qwen3 |

สิ่งที่ควรเห็น: คะแนนแกว่ง**หลายจุด** — มากกว่าช่องว่างระหว่างโมเดลบน leaderboard ทั่วไป
นี่คือคำตอบของคำถามที่ตอนนี้ตั้งไว้: เวลาต่างสำนักรายงานตัวเลขไม่ตรงกัน
ส่วนมาก**ไม่มีใครโกหก** — แค่ไม่มีใครวัดด้วยเครื่องเดียวกัน


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.1 -- ตารางแกว่งคะแนน: โมเดลเดิม การตั้งค่าต่างกันทีละจุด
# -----------------------------------------------------------------------------
import pandas as pd

N_THINK = 10          # ข้อที่ใช้วัดโหมด thinking (ช้ากว่าปกติมาก จึงลดจำนวน)

rows = []


def add_row(label, correct_list, note=""):
    k, n = sum(correct_list), len(correct_list)
    lo, hi = wilson_ci(k, n)
    rows.append({"การตั้งค่า": label, "ถูก/ทั้งหมด": f"{k}/{n}",
                 "accuracy": round(k / n, 3),
                 "ci_low": round(lo, 3), "ci_high": round(hi, 3), "หมายเหตุ": note})
    return k / n


# แถว 1-2: ใช้คะแนน loglik ที่คิดไว้แล้ว (ให้คะแนนครั้งเดียว ตัดสินสองแบบ)
add_row("loglik gamma=1, 0-shot (สัญญาประจำซีรีส์)", anchor_g1)
add_row("loglik gamma=0, 0-shot", anchor_g0, "acc ของ lm-eval")

# แถว 3: 5-shot -- prompt ยาวขึ้น ต้องให้คะแนนใหม่ทั้งชุด
t0 = time.time()
anchor_5shot = [pick(choice_scores(model, tokenizer, it, fewshot=EXAM_FEWSHOT), 1.0)
                == it["answer_idx"] for it in EXAM_ITEMS]
add_row(f"loglik gamma=1, {len(EXAM_FEWSHOT)}-shot", anchor_5shot,
        f"{time.time() - t0:.0f}s")

# แถว 4: generative exact-match บนข้อสอบปรนัย (ถามเป็นแชท ให้ตอบตัวอักษร)
def gen_letter(m, item, thinking=False, max_new=48):
    q = item["question"] + "\n" + "\n".join(
        f"{l}. {c}" for l, c in zip(item["letters"], item["choices"]))
    q += "\nตอบด้วยตัวอักษรของตัวเลือกที่ถูกต้องเพียงตัวเดียว"
    text = generate_answer(m, q, max_new=(384 if thinking else max_new), thinking=thinking)
    mm = re.search(r"\b([a-e])\b", text.lower())
    if mm and mm.group(1) in item["letters"]:
        return item["letters"].index(mm.group(1))
    for i, ch in enumerate(item["choices"]):     # เผื่อโมเดลตอบเป็นข้อความเต็ม
        if normalize_thai(ch) and normalize_thai(ch) in normalize_thai(text):
            return i
    return -1


t0 = time.time()
anchor_gen = [gen_letter(model, it) == it["answer_idx"] for it in EXAM_ITEMS]
add_row("generative exact-match, 0-shot", anchor_gen, f"{time.time() - t0:.0f}s")

# แถว 5-6: ผลจาก GSM8K-TH (โหมด generative vs โหมด judge -- จากหัวข้อ 7.2/7.3)
if GSM_RECORDS and JUDGE_RECORDS:
    add_row("GSM8K-TH: generative exact-match", [r["correct"] for r in GSM_RECORDS])
    add_row("GSM8K-TH: LLM-as-judge", [bool(j["judge_score"]) for j in JUDGE_RECORDS],
            "judge = โมเดลตัวเดียวกัน!")

# แถว 7: เปิด thinking mode (ช้ามาก -- วัดบน subset แล้วบอกตรง ๆ ว่า n เล็กลง)
t0 = time.time()
anchor_think = [gen_letter(model, it, thinking=True) == it["answer_idx"]
                for it in EXAM_ITEMS[:N_THINK]]
add_row(f"generative + enable_thinking=True (n={N_THINK})", anchor_think,
        f"{time.time() - t0:.0f}s -- ช้ากว่าหลายเท่า")

df_swing = pd.DataFrame(rows)
print(df_swing.to_string(index=False))

exam_mask = ~df_swing["การตั้งค่า"].str.startswith("GSM8K")
exam_rows = df_swing[exam_mask].reset_index(drop=True)
swing = exam_rows["accuracy"].max() - exam_rows["accuracy"].min()
print(f"\nคะแนนของโมเดลตัวเดิมบนข้อสอบปรนัยชุดเดิม แกว่ง {100 * swing:.1f} จุด")
print("ตามการตั้งค่าที่ paper ส่วนใหญ่ไม่รายงาน -- เทียบกับช่องว่าง 1-2 จุด")
print("บน leaderboard ที่คนใช้ประกาศผู้ชนะ")

use_thai_font()
fig, ax = plt.subplots(figsize=(9.5, 4.8))
y = np.arange(len(exam_rows))
ax.errorbar(100 * exam_rows["accuracy"], y,
            xerr=[100 * (exam_rows["accuracy"] - exam_rows["ci_low"]),
                  100 * (exam_rows["ci_high"] - exam_rows["accuracy"])],
            fmt="o", capsize=4, color="#2563eb")
ax.set_yticks(y)
ax.set_yticklabels(exam_rows["การตั้งค่า"], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("accuracy (%)")
ax.set_title("โมเดลตัวเดิม น้ำหนักเดิมทุกไบต์ -- คะแนนขึ้นกับวิธีวัด", fontsize=11)
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig("fig_scoring_swing.png", dpi=150, bbox_inches="tight")
plt.show()
print("บันทึกรูปแล้ว: fig_scoring_swing.png")


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.2 -- pass@k แบบไม่ลำเอียง (สมการ 3.3) บนโจทย์เลขที่ตรวจอัตโนมัติได้
# -----------------------------------------------------------------------------
from itertools import combinations as _comb

from kobeval import pass_at_k

# --- self-check กับการนับตรง ๆ ก่อนใช้ (CPU ล้วน) --------------------------------
# n=5 ตัวอย่าง ถูก c=2: pass@2 = 1 - C(3,2)/C(5,2) = 1 - 3/10 = 0.7 พอดี
val = pass_at_k(5, 2, 2)
_all = list(_comb(range(5), 2))
brute = sum(1 for combo in _all if any(i < 2 for i in combo)) / len(_all)
print(f"self-check | pass_at_k(5, 2, 2) = {val:.6f}  นับตรง ๆ ทุก combination = {brute:.6f}")
assert abs(val - 0.7) < 1e-12 and abs(val - brute) < 1e-12
assert pass_at_k(4, 0, 4) == 0.0 and pass_at_k(4, 4, 1) == 1.0
print("self-check | ตรงกับการแจกแจงทุกกรณี -> ผ่าน")

# --- วัดจริง: สุ่ม n คำตอบต่อข้อ (ประกาศชัด: แถวนี้ "ออกนอกสัญญา greedy" โดยตั้งใจ
#     เพราะ pass@k นิยามบนการสุ่ม -- และนี่คือเหตุผลที่มันต้องรายงานแยกจาก acc) -----
N_PASS_ITEMS, N_SAMPLES = 8, 4
PASS_RESULTS = None
if GSM_ITEMS:
    t0 = time.time()
    per_item_c = []
    for i, it in enumerate(GSM_ITEMS[:N_PASS_ITEMS]):
        c = 0
        ids = chat_ids(it["question"] + "\nแสดงวิธีคิดสั้น ๆ แล้วจบด้วยบรรทัด: คำตอบคือ <ตัวเลข>")
        for s in range(N_SAMPLES):
            torch.manual_seed(SEED + s)          # seed ต่างกันต่อ sample, ทำซ้ำได้
            with torch.no_grad():
                out = model.generate(ids, max_new_tokens=160, do_sample=True,
                                     temperature=0.8, top_p=0.95,
                                     pad_token_id=tokenizer.pad_token_id)
            text = tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
            if extract_final_int(thai_digits_to_arabic(text)) == it["gold"]:
                c += 1
        per_item_c.append(c)
        print(f"  ข้อ {i}: ถูก {c}/{N_SAMPLES} sample")

    PASS_RESULTS = {f"pass@{k}": sum(pass_at_k(N_SAMPLES, c, k) for c in per_item_c) / len(per_item_c)
                    for k in (1, 2, 4)}
    print(f"\nGSM8K-TH ({N_PASS_ITEMS} ข้อ, สุ่มข้อละ {N_SAMPLES} ครั้ง, "
          f"{time.time() - t0:.0f}s):")
    for kk, v in PASS_RESULTS.items():
        print(f"  {kk} = {v:.3f}")
    print("pass@1 ควรใกล้ exact-match แบบ greedy | pass@4 สูงกว่า = ความรู้มีอยู่")
    print("แต่การ decode ครั้งเดียวดึงออกมาไม่ได้ -- สองตัวเลขนี้เล่าคนละเรื่อง")
else:
    print("[ข้าม] ไม่มีข้อมูล gsm8k-thai (ดูด่าน license ในหัวข้อ 6)")


### 9.3 Contamination — ข้อสอบรั่วอยู่ในข้อมูลเทรนของเราเองหรือเปล่า

ตอนที่ 1 เราทำ CPT ด้วยคลังเอกสารราชการไทย (`thaigov-v2`) — และ thai_exam
มีข้อสอบเกี่ยวกับความรู้ กฎ และระเบียบของไทย **โอกาสทับซ้อนมีอยู่จริง**
เซลล์ถัดไปสแกนตามสมการ 3.5 ด้วย character n-gram สองระดับ:

- **k=5** — ไวมาก: วลีไทยสั้น ๆ ทั่วไปก็ชนกันได้ ใช้ดูภาพรวมการกระจาย
- **k=20** — เข้มงวด: 20 ตัวอักษรติดกันที่ตรงเป๊ะแทบไม่เกิดโดยบังเอิญ
  ข้อไหนมี overlap@20 สูง คือสัญญาณ "เคยเห็นเฉลย" ที่ควรตัดออกจากการสรุปผล

ค่า $k$ เป็นการตัดสินใจของผู้วัดที่เปลี่ยนผลลัพธ์ทั้งกระดาน — เหมือน $\gamma$
ในหัวข้อ 7.1 — จึงต้องรายงานทั้งคู่ ไม่ใช่เลือกอันที่ทำให้ตัวเองดูสะอาด
ถ้าเจอ hit ให้**รายงานรายข้อและตัดออกจากการสรุปผล** ไม่ใช่ทำเป็นมองไม่เห็น


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.3 -- contamination scan: thai_exam <-> คลัง CPT ของตอนที่ 1 (thaigov)
# -----------------------------------------------------------------------------
N_CORPUS_DOCS = 1200      # จำนวนเอกสาร thaigov ที่สแกน (เท่าที่ RAM ของ Colab ไหว)
DOC_CLIP = 1500           # ตัดเอกสารละไม่เกินกี่ตัวอักษร


def char_ngrams(text, k):
    text = re.sub(r"\s+", "", text)      # ตัดช่องว่างก่อน -- ไทยไม่ใช้ช่องว่างแบ่งคำ
    return {text[i:i + k] for i in range(len(text) - k + 1)} if len(text) >= k else set()


# self-check เล็ก ๆ ของฟังก์ชันก่อนใช้จริง
assert char_ngrams("กขคง", 2) == {"กข", "ขค", "คง"}
assert char_ngrams("ก ข ค ง", 2) == {"กข", "ขค", "คง"}, "ต้องตัดช่องว่างก่อนหั่น n-gram"

corpus_raw = load_dataset("pythainlp/thaigov-v2-corpus-22032023", split="train")
corpus_raw = corpus_raw.shuffle(seed=SEED).select(range(min(N_CORPUS_DOCS, len(corpus_raw))))
text_col = next(c for c in corpus_raw.column_names
                if c.lower() in ("text", "context", "content", "body", "article", "document"))

t0 = time.time()
G5, G20 = set(), set()
for doc in corpus_raw[text_col]:
    if isinstance(doc, str):
        G5 |= char_ngrams(doc[:DOC_CLIP], 5)
        G20 |= char_ngrams(doc[:DOC_CLIP], 20)
print(f"สร้างดัชนี n-gram จาก thaigov {len(corpus_raw)} เอกสาร: "
      f"|G5|={len(G5):,}  |G20|={len(G20):,}  ({time.time() - t0:.0f}s)")

scan = []
for i, it in enumerate(EXAM_ITEMS):
    text = it["question"] + " " + " ".join(it["choices"])
    g5, g20 = char_ngrams(text, 5), char_ngrams(text, 20)
    hit20 = g20 & G20
    scan.append({
        "item": i,
        "overlap5": len(g5 & G5) / len(g5) if g5 else 0.0,
        "overlap20": len(hit20) / len(g20) if g20 else 0.0,
        "example_hit20": next(iter(hit20)) if hit20 else None,
    })

ov5 = [s["overlap5"] for s in scan]
ov20 = [s["overlap20"] for s in scan]
print(f"\noverlap@5  : mean={sum(ov5) / len(ov5):.3f}  max={max(ov5):.3f}")
print(f"overlap@20 : mean={sum(ov20) / len(ov20):.3f}  max={max(ov20):.3f}")
print("(overlap@5 สูงเป็นเรื่องปกติของภาษาเดียวกัน -- ตัวชี้วัดจริงคือ @20)")

FLAG_THRESHOLD = 0.10
flagged = [s for s in scan if s["overlap20"] >= FLAG_THRESHOLD]
print(f"\nข้อที่ overlap@20 >= {FLAG_THRESHOLD:.0%}: {len(flagged)} ข้อ")
for s in sorted(scan, key=lambda x: -x["overlap20"])[:5]:
    print(f"  ข้อ {s['item']:2d}: overlap20={s['overlap20']:.3f} overlap5={s['overlap5']:.3f}"
          + (f"  ตัวอย่าง gram ที่ชน: {s['example_hit20']!r}" if s["example_hit20"] else ""))

if flagged:
    clean = [i for i in range(len(EXAM_ITEMS)) if i not in {s["item"] for s in flagged}]
    k_all, n_all = sum(anchor_g1), len(anchor_g1)
    k_cln = sum(anchor_g1[i] for i in clean)
    lo_a, hi_a = wilson_ci(k_all, n_all)
    lo_c, hi_c = wilson_ci(k_cln, len(clean))
    print(f"\nanchor บนข้อสอบทั้งชุด : {k_all}/{n_all} = {k_all / n_all:.3f} [CI {lo_a:.3f}-{hi_a:.3f}]")
    print(f"anchor เมื่อตัดข้อที่ flag: {k_cln}/{len(clean)} = {k_cln / len(clean):.3f} "
          f"[CI {lo_c:.3f}-{hi_c:.3f}]")
    print("รายงานทั้งสองตัวเลขเสมอ -- อย่าเลือกตัวที่สวยกว่า")
else:
    print("\nไม่มีข้อไหนแตะเกณฑ์ overlap@20 -- ข้อสอบชุดนี้กับคลัง CPT ตัวอย่าง")
    print("ไม่พบการรั่วระดับวลียาว (แต่จำไว้: เราสแกนแค่ตัวอย่างของคลัง ไม่ใช่ทั้งคลัง)")

CONTAMINATION = {"k5_mean": sum(ov5) / len(ov5), "k5_max": max(ov5),
                 "k20_mean": sum(ov20) / len(ov20), "k20_max": max(ov20),
                 "n_flagged_at_10pct": len(flagged),
                 "corpus_docs_scanned": len(corpus_raw)}


In [ ]:
# -----------------------------------------------------------------------------
# หัวข้อ 9.4 -- รวมทุกตัวเลขลง results.json ไฟล์เดียว
# -----------------------------------------------------------------------------
from kobeval import write_results

payload = {
    "post": 9,
    "slug": "llm-09-benchmarking",
    "notebook": "09_benchmarking.ipynb",
    "model": MODEL_ID,
    "gpu": GPU_NAME,
    "supports_bf16": SUPPORTS_BF16,
    "eval_contract": EVAL_CONTRACT,
    "licenses": {"typhoon-ai/thai_exam": EXAM_LICENSE, "VISAI-AI/gsm8k-thai": GSM_LICENSE,
                 "gates_passed": {"thai_exam": EXAM_OK, "gsm8k_thai": GSM_OK}},
    "exam_source": EXAM_SOURCE,
    "n_exam_items": len(EXAM_ITEMS),
    "kobeval_baseline": {
        "slices": {k: {kk: v[kk] for kk in ("accuracy", "ci_low", "ci_high",
                                            "n_correct", "n", "th_ratio")}
                   for k, v in baseline["slices"].items()},
        "overall": baseline["overall"],
    },
    "checkpoints": {label: {k: v for k, v in rec.items() if k != "correct"}
                    for label, rec in SWEEP.items()},
    "mcnemar_closest_pair": {"pair": [A, B], "a": a, "b": b, "c": c, "d": d, **test},
    "scoring_swing": rows,
    "swing_points": round(100 * float(swing), 1),
    "pass_at_k": PASS_RESULTS,
    "judge": {"model": JUDGE_MODEL_NAME, "rubric": JUDGE_RUBRIC,
              "self_enhancement_warning": "judge กับผู้ถูกวัดเป็นโมเดลตัวเดียวกัน"},
    "contamination": CONTAMINATION,
    "figures": ["fig_ci_width.png", "fig_all_checkpoints_ci.png", "fig_scoring_swing.png"],
}

path = write_results(payload, "results.json")
print(f"เขียนไฟล์แล้ว: {path}")
print(json.dumps({k: payload[k] for k in ("checkpoints", "swing_points", "pass_at_k")},
                 ensure_ascii=False, indent=2, default=str))


---

## 10. สรุป (Summary)

- **accuracy ที่ไม่มี CI คือข่าวลือ** — n=100 ให้ ±10 จุด และช่องว่างบน leaderboard
  ส่วนใหญ่เล็กกว่านั้น (`fig_ci_width.png` คือหลักฐาน)
- **Wilson ไม่ใช่ความหรูหรา** — normal approximation ให้ CI กว้างศูนย์ที่ $\hat p = 0$
  ซึ่งเป็นจุดที่เราต้องการมันที่สุด
- **คะแนนเป็นคุณสมบัติของ (โมเดล × วิธีวัด × ข้อสอบ × n)** — เราเพิ่งเห็นโมเดลตัวเดิม
  แกว่งหลายจุดจากแค่ $\gamma$, จำนวน shot, โหมดวัด และ thinking mode
- **เทียบสองโมเดลบนข้อสอบชุดเดียวกัน ให้ใช้ McNemar** — ข้อที่เห็นตรงกันไม่ใช่หลักฐาน
  และคู่ที่คะแนนใกล้กันมักจบที่ "ยังบอกไม่ได้" ซึ่งเป็นคำตอบที่ซื่อสัตย์
- **pass@k ต้องใช้ตัวประมาณทวินามของ kobeval** — ตัวประมาณ plug-in ลำเอียงเพราะ Jensen
- **สแกน contamination ก่อนเชื่อคะแนน** — โดยเฉพาะเมื่อข้อมูลเทรน (thaigov)
  กับข้อสอบ (thai_exam) มาจากโดเมนใกล้กัน และรายงานทั้ง k=5 กับ k=20
- **ด่านตรวจ license ไม่ใช่พิธีกรรม** — การวัดผลที่เริ่มจากการละเมิดเงื่อนไขข้อมูล
  ไม่มีทางเรียกว่า rigorous

**ตอนต่อไป (ตอนจบ):** Deployment — โมเดลที่วัดผลแล้วต้องออกไปเจอผู้ใช้จริง
เราจะคำนวณเพดานความเร็วจาก datasheet ของ T4 ก่อนเขียนโค้ดแม้แต่บรรทัดเดียว
แล้ววัดของจริงมาเทียบ


---

## ข้อจำกัดของการทดลองนี้

เขียนไว้ตรง ๆ เพื่อไม่ให้ตัวเลขข้างบนถูกอ่านเกินกว่าที่มันบอกได้จริง

1. **ชุดข้อสอบที่ใช้เล็กมาก** — thai_exam เราใช้ 60 ข้อ (จากชุดเต็มหลายพันข้อ)
   และ gsm8k-thai แค่ 20 ข้อ เพราะงบเวลา T4 ฟรี CI จึงกว้างมากตามกราฟในหัวข้อ 4
   ตัวเลขทั้งหมดมีไว้ **สอนวิธีวัด** ไม่ใช่เพื่อจัดอันดับโมเดล

2. **KobEval-TH มี slice ละ 30 ข้อ** — ความต่างระดับ 1-2 ข้อไม่ใช่ความต่าง
   ที่มีนัยสำคัญ และ `th_ratio` เป็นค่าเฉลี่ยที่ไม่มีการทดสอบนัยสำคัญกำกับ

3. **Judge คือโมเดลตัวเดียวกับผู้ถูกวัด** — ผิดหลักการโดยตั้งใจเพื่อสาธิต
   self-enhancement bias บนเครื่องฟรี ตัวเลขจาก judge ในตอนนี้ห้ามนำไปอ้างอิง

4. **Contamination scan ครอบคลุมแค่ตัวอย่างของคลัง** — เราสแกน thaigov 1,200 เอกสาร
   จากคลังเต็ม การไม่เจอ hit ไม่ได้พิสูจน์ความสะอาด มันแปลว่า "ยังไม่เจอในตัวอย่างนี้"
   และเราไม่มีทางสแกนข้อมูล pretrain ของ Qwen3 เองได้เลย

5. **ใช้ thai_exam เพียง subset เดียว (onet)** — ชุดเต็มมี 5 วิชา วิธีเฉลี่ยข้ามวิชา
   ก็เป็นการตัดสินใจที่เปลี่ยนคะแนนได้อีกชั้น

6. **โมเดลเล็ก** — Qwen3-0.6B อยู่ใกล้เส้นเดาสุ่มบนข้อสอบยาก ๆ ผลแกว่งของวิธีวัด
   บนโมเดลใหญ่จะมีขนาดต่างจากนี้ (แต่ทิศทางของบทเรียนเหมือนกัน)

7. **รันครั้งเดียว seed เดียว** — ตามสัญญาของซีรีส์เพื่อให้ทำซ้ำได้
   แต่การกวาดหลาย seed จะให้ภาพความแปรปรวนที่สมบูรณ์กว่า

---

*ซีรีส์นี้เผยแพร่ภายใต้สัญญาอนุญาต [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/) — ใช้ต่อได้โดยอ้างอิงที่มา ห้ามใช้เชิงพาณิชย์ และต้องเผยแพร่ต่อด้วยสัญญาเดียวกัน — [kobkrit.com](https://kobkrit.com)*
